In [23]:
import sys
print(sys.executable)

c:\Users\divya\Desktop\AI\LangChain_Lab\.venv\Scripts\python.exe


In [12]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from common.llm import get_llm

In [13]:
print(type(get_llm()))

<class 'langchain_groq.chat_models.ChatGroq'>


In [15]:
llm = get_llm()

print(llm.metadata["lc_versions"])

{'langchain-core': '1.6.4', 'langchain': '1.4.2'}


In [17]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


In [18]:
prompt  = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("chat_history"),
    ("user", "{input}"),
])


In [19]:
model = get_llm(temperature=0.7)
chain =  prompt | model

In [20]:
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [25]:
chain_with_history = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

C:\Users\divya\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [26]:
config = {"configurable": {"session_id": "user_123"}}

In [27]:
response = chain_with_history.invoke(
    {"input" : "Hi! My name is Divyanshu"},
    config = config,
)

print(response)

C:\Users\divya\AppData\Local\Temp\ipykernel_21140\2687409421.py:5: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = InMemoryChatMessageHistory()


content='Hello, Divyanshu! 👋 How can I help you today?' additional_kwargs={'reasoning_content': 'We need to respond in a friendly manner. The user just introduced themselves. We should greet and ask how we can help.'} response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 88, 'total_tokens': 138, 'completion_time': 0.054913326, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.005088071, 'prompt_tokens_details': None, 'queue_time': 0.315841234, 'total_time': 0.060001397}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0d877-2386-7b11-809e-0bb67f8741d1-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 88, 'output_tokens': 50, 'total_tokens': 138, 'output_token_details': {'reasoning': 26}}


In [28]:
for m in store["user_123"].messages:
    print(f"[{m.type}] {m.content}")

[human] Hi! My name is Divyanshu
[ai] Hello, Divyanshu! 👋 How can I help you today?


In [31]:
config1 = {"configurable": {"session_id": "user_123"}}

response3 = chain_with_history.invoke(
    {"input": "What is my name?"},
    config=config1,
)
print(response3.content)

Your name is Divyanshu.


In [32]:
config2 = {"configurable": {"session_id": "user_456"}}

response3 = chain_with_history.invoke(
    {"input": "What is my name?"},
    config=config2,
)
print(response3.content)

I’m sorry, but I don’t have that information.


In [34]:
config = {"configurable": {"session_id": "user_123"}}

for chunk in chain_with_history.stream(
    {"input": "Tell me a one-line fun fact about my name."},
    config=config,
):
    print(chunk.content, end="", flush=True)

Did you know that “Divyanshu” literally means “divine light” in Sanskrit, and it’s also the name of a rising Indian cricketer who’s been making waves in international matches?

In [35]:
for m in store["user_123"].messages:
    print(f"[{m.type}] {m.content}")

[human] Hi! My name is Divyanshu
[ai] Hello, Divyanshu! 👋 How can I help you today?
[human] What is my name?
[ai] Your name is Divyanshu.
[human] What is my name?
[ai] Your name is Divyanshu.
[human] Tell me a one-line fun fact about my name.
[AIMessageChunk] Fun fact: “Divyanshu” is a Sanskrit name meaning “full of divine light,” and it’s also the first name of the popular Indian cricketer Divyanshu Sharma.
[human] Tell me a one-line fun fact about my name.
[AIMessageChunk] Did you know that “Divyanshu” literally means “divine light” in Sanskrit, and it’s also the name of a rising Indian cricketer who’s been making waves in international matches?
